# Feature-Engineering Validation — before/after distributions + augmented samples

**Goal:** verify the feature engineering (#20–#22) did what §5 intended, with
before/after evidence:

1. **Normalization** — the transformed set has the expected range/dtype (`[0,1]`
   float for rescale; ~0 mean / ~1 std for standardize).
2. **Equalization** — the post-equalization histogram is **flatter** (higher entropy).
3. **Augmentation** — flips/rotations/zooms/shifts stay within "still a natural face"
   (label-preserving), and keep shape/dtype.

These checks catch mistakes early: double-normalization, wrong dtype, or
over-aggressive augmentation. All use the **real** modules the training pipeline
uses (`build_normalizer`, `build_augmenter`). Sampling is seeded.

## 0. Setup

In [ ]:
import sys
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config
from src.emotion_detector.utils.logging import setup_logging
from src.emotion_detector.data.fer2013 import Fer2013Fetcher
from src.emotion_detector.data.preprocessing import build_normalizer
from src.emotion_detector.data.augmentation import build_augmenter

cfg = load_config(ROOT / "config.yaml")
for key in cfg["paths"]:
    cfg["paths"][key] = str(ROOT / cfg["paths"][key])
setup_logging(cfg)

SEED = cfg["global"]["seed"]
rng = np.random.default_rng(SEED)
EDA_DIR = Path(cfg["paths"]["results_dir"]) / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

EMOTION_LABELS = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad",     5: "Surprise", 6: "Neutral",
}


def entropy(img):
    """Shannon entropy of the 256-bin intensity histogram (higher = flatter)."""
    counts, _ = np.histogram(img, bins=256, range=(0, 256))
    p = counts[counts > 0] / counts.sum()
    return float(-(p * np.log2(p)).sum())


def cfg_with(**preproc):
    """Copy of cfg with preprocessing overrides."""
    c = copy.deepcopy(cfg)
    c["preprocessing"].update(preproc)
    return c

## 1. Load the raw training images

In [ ]:
X_raw, y_raw = Fer2013Fetcher(cfg).fetch("Training")
print(f"raw: {X_raw.shape} {X_raw.dtype}, range [{X_raw.min()}, {X_raw.max()}]")

## 2. Normalization — pixel distribution before vs after

`rescale` should map `[0, 255]` uint8 → `[0, 1]` float32. The histogram keeps its
*shape* but its x-axis compresses to `[0, 1]`.

In [ ]:
norm = build_normalizer(cfg_with(normalization="rescale"))
X_rescaled = norm.fit(X_raw).transform(X_raw)

print(f"rescaled: {X_rescaled.dtype}, range [{X_rescaled.min():.3f}, {X_rescaled.max():.3f}]")
assert X_rescaled.dtype == np.float32, "expected float32"
assert X_rescaled.min() >= 0.0 and X_rescaled.max() <= 1.0, "expected [0, 1]"
print("✓ rescale → float32 in [0, 1]")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(X_raw.reshape(-1), bins=64, color="steelblue")
axes[0].set_title("Before — raw uint8 [0, 255]"); axes[0].set_xlabel("intensity")
axes[1].hist(X_rescaled.reshape(-1), bins=64, color="seagreen")
axes[1].set_title("After — rescaled float [0, 1]"); axes[1].set_xlabel("intensity")
plt.tight_layout()
fig.savefig(EDA_DIR / "fe_normalization_hist.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'fe_normalization_hist.png'}")
plt.show()

## 3. Standardize — zero-mean / unit-variance check

`standardize` uses **train** mean/std; the standardized train set should have
mean ≈ 0 and std ≈ 1.

In [ ]:
std_norm = build_normalizer(cfg_with(normalization="standardize"))
X_std = std_norm.fit(X_raw).transform(X_raw)
print(f"standardized: mean={X_std.mean():.4f}, std={X_std.std():.4f}")
assert abs(X_std.mean()) < 1e-3 and abs(X_std.std() - 1.0) < 1e-3
print("✓ standardize → mean ~0, std ~1 (train statistics)")

## 4. Equalization — flatter histogram

Global histogram equalization remaps intensities via the CDF, so the output
histogram is **flatter** (closer to uniform → higher entropy). We compare the raw
histogram to the equalized one (converted back to uint8 for a like-for-like plot).

In [ ]:
eq_norm = build_normalizer(cfg_with(normalization="histogram_eq"))
X_eq = eq_norm.fit(X_raw).transform(X_raw)          # float [0, 1] (equalize → rescale)
X_eq_u8 = np.clip(X_eq * 255, 0, 255).astype(np.uint8)

e_raw, e_eq = entropy(X_raw), entropy(X_eq_u8)
print(f"histogram entropy: raw={e_raw:.3f}  →  equalized={e_eq:.3f}")
assert e_eq > e_raw, "equalized histogram should be flatter (higher entropy)"
print("✓ equalization flattened the histogram")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(X_raw.reshape(-1), bins=64, range=(0, 255), color="slategray")
axes[0].set_title(f"Before — raw (entropy {e_raw:.2f})")
axes[1].hist(X_eq_u8.reshape(-1), bins=64, range=(0, 255), color="indianred")
axes[1].set_title(f"After — histogram_eq (entropy {e_eq:.2f})")
for ax in axes:
    ax.set_xlabel("intensity")
plt.tight_layout()
fig.savefig(EDA_DIR / "fe_equalization_hist.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'fe_equalization_hist.png'}")
plt.show()

## 5. Augmentation — original vs augmented faces

Render the same faces alongside a few augmented versions. Confirm each augmented
face is still a **plausible, correctly-labelled** face (label-preserving) — no
upside-down or wildly warped results.

*Requires TensorFlow (the augmenter). Augmentation runs on normalized `[0,1]` floats.*

In [ ]:
augmenter = build_augmenter(cfg)  # seeded from global.seed

n_faces, n_aug = 5, 3
idx = rng.choice(len(X_rescaled), size=n_faces, replace=False)
faces = X_rescaled[idx][..., None]  # (n_faces, 48, 48, 1)

fig, axes = plt.subplots(n_faces, n_aug + 1, figsize=((n_aug + 1) * 1.8, n_faces * 1.8))
for r in range(n_faces):
    axes[r, 0].imshow(faces[r, ..., 0], cmap="gray", vmin=0, vmax=1)
    axes[r, 0].set_ylabel(EMOTION_LABELS[int(y_raw[idx[r]])], rotation=0, ha="right",
                          va="center", fontsize=9)
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    if r == 0:
        axes[r, 0].set_title("original", fontsize=9)
    for c in range(1, n_aug + 1):
        aug_img = augmenter(faces[r:r + 1], training=True)
        axes[r, c].imshow(np.asarray(aug_img)[0, ..., 0], cmap="gray", vmin=0, vmax=1)
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(f"aug {c}", fontsize=9)
fig.suptitle("Original vs augmented (label-preserving)", fontsize=12)
plt.tight_layout()
fig.savefig(EDA_DIR / "fe_augmented_samples.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'fe_augmented_samples.png'}")
plt.show()

In [ ]:
# Augmentation must preserve shape and stay in range; labels are never touched.
batch = X_rescaled[:16][..., None]
aug_batch = np.asarray(augmenter(batch, training=True))
print(f"augmented batch: {aug_batch.shape}, range [{aug_batch.min():.2f}, {aug_batch.max():.2f}]")
assert aug_batch.shape == batch.shape, "augmentation must preserve shape"
print("✓ augmentation preserves shape; labels ride along unchanged")

## 6. FE stage ON vs OFF

With `stages.preprocessing` off, `build_normalizer` returns the identity → raw
pixels pass through unchanged. This is the raw-pixel baseline the ablation compares
against.

In [ ]:
cfg_off = copy.deepcopy(cfg)
cfg_off["stages"]["preprocessing"] = False
off = build_normalizer(cfg_off)
X_off = off.fit(X_raw).transform(X_raw)

print(f"FE OFF: {X_off.dtype}, range [{X_off.min():.1f}, {X_off.max():.1f}] (raw pixels as float)")
print(f"FE ON : {X_rescaled.dtype}, range [{X_rescaled.min():.3f}, {X_rescaled.max():.3f}]")
assert X_off.max() > 1.0, "stage OFF should leave raw [0, 255] values"
print("✓ stage OFF = raw-pixel passthrough; stage ON = normalized")

## 7. Findings — FE validation

| Check | Expected | Result |
|---|---|---|
| `rescale` range / dtype | `[0, 1]` float32 | ✓ asserted |
| `standardize` mean / std | ~0 / ~1 | ✓ asserted |
| `histogram_eq` histogram | flatter (entropy ↑) | ✓ asserted |
| Augmentation shape | preserved `(N, 48, 48, 1)` | ✓ asserted |
| Augmented faces | still natural, correct label | ✓ visual (§5) |
| FE stage OFF | raw-pixel passthrough | ✓ asserted |

Every numeric expectation is `assert`-ed, so a clean run proves the FE stage does
exactly what `data.md §5` claims. No double-normalization, no dtype drift, no
over-aggressive augmentation observed.